## 0. Setup (run first, every session)
**Before you start**
1. *Kaggle:* Settings -> Accelerator = **GPU T4/P100**, Internet = **On**; Add-ons -> Secrets -> add `HF_TOKEN` (a Hugging Face token with *write* access). *Colab:* Runtime -> GPU, and add `HF_TOKEN` under the key icon (Secrets), notebook access on.
2. Edit `HF_REPO` below (e.g. `yourname/atml-pa3-backup`). A **private** repo is created automatically on the first push.
3. A background loop pushes results + checkpoints to that repo every 10 minutes, and `pull` restores them in a fresh session. **If a session dies, re-run section 0 and then re-run the same training cell: it resumes from the last checkpoint.**

4. **Not yet run end to end.** Offline steps (3.1-3.3, 3.5) are CPU-only and can run locally; only the live generation (3.4) needs a GPU.
5. Provide the supplied rollout data under `data/ATML-PA3-Data/` (download with the id from LMS, cell 3.0).

In [ ]:
import os, sys, subprocess, threading, time
HF_REPO  = "YOUR_USERNAME/atml-pa3-backup"      # <- edit
REPO_URL = "https://github.com/ZainFarhan-Work/ATML.git"
BRANCH   = "main"

ON_KAGGLE = os.path.exists("/kaggle")
ROOT = "/kaggle/working" if ON_KAGGLE else "/content"
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
else:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

if not os.path.exists(f"{ROOT}/ATML"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, f"{ROOT}/ATML"], check=True)
else:
    subprocess.run(["git", "-C", f"{ROOT}/ATML", "pull"], check=False)
os.chdir(f"{ROOT}/ATML/PA 3")
print(os.getcwd())

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m scripts.check_environment

In [ ]:
# restore anything saved by a previous session (harmless on the first run)
r = subprocess.run([sys.executable, "-m", "scripts.sync_hub", "pull", "--repo", HF_REPO, "--task", "task3"])
print("pull exit code:", r.returncode)

# background backup every 10 min (and call sync_now() manually after each section)
def sync_now():
    subprocess.run([sys.executable, "-m", "scripts.sync_hub", "push", "--repo", HF_REPO, "--task", "task3"])
def _loop():
    while True:
        time.sleep(600)
        try: sync_now()
        except Exception as e: print("sync failed:", e)
threading.Thread(target=_loop, daemon=True).start()

## 3.0 Data

In [ ]:
# !python -m scripts.download_assets --task3-repo <hf-dataset-id-from-LMS>
!python -m scripts.inspect_task3_data

## 3.1 Repeated sampling, scoring and aggregation (Steps 1-2)

In [ ]:
for ds in ["math", "mathvista"]:
    !python -m task3_ttc.offline --dataset {ds} --workers 4
sync_now()

## 3.2 What the signals capture (Step 3)

In [ ]:
for ds in ["math", "mathvista"]:
    !python -m task3_ttc.signals --dataset {ds}

## 3.3 Parallel vs sequential, live generation (Step 4) - GPU

In [ ]:
!python -m task3_ttc.live
sync_now()

## 3.4 Adaptive sampling (Step 5)

In [ ]:
for ds in ["math", "mathvista"]:
    !python -m task3_ttc.adaptive --dataset {ds}
sync_now()

## 3.5 Compute allocation and failure cases (Step 6)

In [ ]:
!python -m task3_ttc.analyze
sync_now()